# 01_pipeline
Captured 2026-11-21 22:52 UTC

**Raw files to governed gold.** Generator wrote 47748 PO lines with 954 planted duplicates, 1441 lines in grams and 348 negative forecasts; the pipeline quarantined the duplicates and negatives, filled missing lead times from the source list and matched every supplier quality code to an ERP vendor.


In [1]:
-- generator run summary (run_log/generate_base.json)
-- src/data/generate.py

{
 "task": "generate",
 "scenario": "base",
 "as_of": "2026-11-20",
 "seed": 42,
 "vendors": 60,
 "materials": 120,
 "source_list": 239,
 "single_source_api_pct": 37.5,
 "skus": 297,
 "bom_lines": 1035,
 "po_lines": 47748,
 "po_duplicates_planted": 954,
 "po_lines_in_grams": 1441,
 "po_lines_no_lead_time": 2980,
 "inventory_rows": 41850,
 "quality_events": 377,
 "forecast_files": 9,
 "negative_forecasts": 348
}

In [2]:
-- pipeline expectations, most recent check per rule (from the pipeline event log)
SELECT ... FROM event_log(<pipeline_id>)

expectation      | dataset                 | on_violation | passed | failed | last_checked            
---------------------------------------------------------------------------------------------------------
schema_contract  | bronze.supplier_event   | WARN         | 0      | 1      | 2026-11-21T22:44:40.342Z
non_negative     | silver.forecast         | DROP ROW     | 9462   | 42     | 2026-11-21T22:47:59.628Z
has_lead_time    | silver.po_line          | WARN         | 44768  | 2980   | 2026-11-21T22:47:59.540Z
po_key           | silver.po_line          | FAIL UPDATE  | 47748  | 0      | 2026-11-21T22:47:59.540Z
feed_on_time     | silver.source_freshness | WARN         | 4      | 0      | 2026-11-21T22:48:13.126Z
matched_supplier | silver.supplier_event   | WARN         | 380    | 0      | 2026-11-21T22:48:05.680Z
vendor_key       | silver.vendor           | FAIL UPDATE  | 60     | 0      | 2026-11-21T22:47:54.587Z
(7 rows)

In [3]:
-- schema contract: an unexpected field is rescued, not added (test file with inspectorName)
SELECT supplierCode, _rescued_data FROM supply_chain.bronze.supplier_event WHERE _rescued_data IS NOT NULL

supplierCode | eventType | _rescued_data               
------------------------------------------------------------------------------------------------------------------------
SQ-0112      | AUDIT     | {"inspectorName":"synthetic 
(1 rows)

In [4]:
-- feed freshness vs expected cadence (warn and track late or missing feeds)
SELECT * FROM supply_chain.silver.source_freshness

feed             | latest     | max_age_days | age_days | status 
--------------------------------------------------------------------
demand_forecast  | 2026-11-20 | 8            | 0        | ON_TIME
erp_inventory    | 2026-11-20 | 1            | 0        | ON_TIME
erp_po_lines     | 2026-11-19 | 3            | 1        | ON_TIME
supplier_quality | 2026-11-20 | 30           | 0        | ON_TIME
(4 rows)

In [5]:
-- row counts and data quality by layer
SELECT ... FROM bronze/silver/gold

t                            | n    
-------------------------------------------------
silver.po_line lead time fil | 2980 
silver.supplier_event unmatc | 0    
gold.api_exposure            | 288  
gold.days_of_cover           | 288  
gold.disruption_impact       | 38   
bronze.po_line               | 48702
silver.po_line               | 47748
silver.po_line_quarantine (d | 954  
silver.forecast_quarantine ( | 42   
(9 rows)

In [6]:
-- supplier code conformance
SELECT match_method, count(*) FROM supply_chain.silver.supplier_xref GROUP BY 1

match_method | n 
--------------------
DUNS         | 60
(1 rows)

In [7]:
-- file-arrival trigger: a new supplier quality file starts the job by itself
databricks jobs list-runs --job-id <radar_main>

[
 {
  "run_id": 691170754805340,
  "trigger": "FILE_ARRIVAL",
  "result": "SUCCESS",
  "tasks": [
   "explain_impact",
   "generate",
   "land_decisions",
   "pipeline",
   "refresh_gold",
   "score_po_risk",
   "should_generate",
   "sync_lakebase"
  ]
 }
]

In [8]:
-- latest forecast file wins (late and re-sent files ignored)
SELECT published_at, count(*) FROM silver.forecast GROUP BY 1

published_at | n   
----------------------
2026-11-20   | 7454
2026-11-13   | 288 
2026-11-06   | 288 
2026-10-30   | 288 
(4 rows)